# Mental Health Condition Predictor
### From Raw Lifestyle Data to Explainable AI

**Dataset:** Mental Health and Lifestyle Habits (2019–2024) — Kaggle  
**Task:** Multi-class Classification — predict which mental health condition (if any) a person's lifestyle is associated with  
**Classes:** `None`, `Anxiety`, `Depression`, `Bipolar`, `PTSD`  

---
**Workflow:**
1. Load & understand data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Baseline model (Logistic Regression)
5. Ensemble models (Random Forest + XGBoost)
6. Model comparison
7. SHAP explainability
8. Save best model

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import xgboost as xgb
import shap
import joblib

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('All libraries loaded!')

ModuleNotFoundError: No module named 'xgboost'

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/raw/Mental_Health_Lifestyle_Dataset.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include='all').round(2)

In [ ]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing cells: {df.isnull().sum().sum()}')

## 3. Exploratory Data Analysis (EDA)

**Key questions we want to answer:**
- How balanced is the target variable?
- Do people with Depression sleep less?
- Does high screen time correlate with worse mental health?
- Which features are most correlated with each other?

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(9, 4))
order = df['Mental Health Condition'].value_counts().index
bars = sns.countplot(data=df, x='Mental Health Condition', order=order, palette='muted', ax=ax)
ax.set_title('Distribution of Mental Health Conditions', fontsize=14, fontweight='bold')
ax.set_xlabel('Condition')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()
print(df['Mental Health Condition'].value_counts())

In [ ]:
# Stress Level distribution + breakdown by condition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='Stress Level', order=['Low', 'Moderate', 'High'],
              palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Level Distribution', fontsize=13, fontweight='bold')

cross = pd.crosstab(df['Stress Level'], df['Mental Health Condition'], normalize='index') * 100
cross.loc[['Low', 'Moderate', 'High']].plot(kind='bar', ax=axes[1], colormap='tab10', width=0.7)
axes[1].set_title('Condition Breakdown by Stress Level (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Stress Level')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Sleep Hours and Screen Time by condition
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = df['Mental Health Condition'].value_counts().index

sns.boxplot(data=df, x='Mental Health Condition', y='Sleep Hours',
            order=order, palette='muted', ax=axes[0])
axes[0].set_title('Sleep Hours by Mental Health Condition', fontsize=13, fontweight='bold')
axes[0].axhline(y=8, color='red', linestyle='--', alpha=0.5, label='Recommended (8h)')
axes[0].legend()

sns.boxplot(data=df, x='Mental Health Condition', y='Screen Time per Day (Hours)',
            order=order, palette='Set2', ax=axes[1])
axes[1].set_title('Screen Time by Mental Health Condition', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Happiness Score by condition
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x='Mental Health Condition', y='Happiness Score',
            order=order, palette='coolwarm_r', ax=axes[0])
axes[0].set_title('Happiness Score by Condition', fontsize=13, fontweight='bold')

sns.boxplot(data=df, x='Mental Health Condition', y='Social Interaction Score',
            order=order, palette='viridis', ax=axes[1])
axes[1].set_title('Social Interaction Score by Condition', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['Age', 'Sleep Hours', 'Work Hours per Week',
                'Screen Time per Day (Hours)', 'Social Interaction Score', 'Happiness Score']

corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key group statistics
print('=== AVERAGE STATS BY MENTAL HEALTH CONDITION ===')
summary = df.groupby('Mental Health Condition')[numeric_cols].mean().round(2)
print(summary.to_string())

In [ ]:
# Exercise Level breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ex_cross = pd.crosstab(df['Exercise Level'], df['Mental Health Condition'], normalize='index') * 100
ex_cross.loc[['Low', 'Moderate', 'High']].plot(kind='bar', ax=axes[0], colormap='tab10', width=0.7)
axes[0].set_title('Condition by Exercise Level (%)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)

diet_cross = pd.crosstab(df['Diet Type'], df['Mental Health Condition'], normalize='index') * 100
diet_cross.plot(kind='bar', ax=axes[1], colormap='tab10', width=0.7)
axes[1].set_title('Condition by Diet Type (%)', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)

plt.tight_layout()
plt.show()

## 4. Feature Engineering

We create three new features that better capture the lifestyle story:
- **`sleep_debt`** — how many hours short of the recommended 8h  
- **`screen_overload`** — binary flag: screen time > 6 hours/day  
- **`work_intensity`** — work hours relative to a standard 40h week

In [ ]:
df_model = df.copy()

# Engineered features
df_model['sleep_debt'] = np.maximum(0, 8 - df_model['Sleep Hours'])
df_model['screen_overload'] = (df_model['Screen Time per Day (Hours)'] > 6).astype(int)
df_model['work_intensity'] = df_model['Work Hours per Week'] / 40

# Ordinal encoding (Low < Moderate < High preserves order)
oe = OrdinalEncoder(categories=[['Low', 'Moderate', 'High']])
df_model['exercise_enc'] = oe.fit_transform(df_model[['Exercise Level']])
df_model['stress_enc'] = oe.fit_transform(df_model[['Stress Level']])

# Label encoding for other categoricals
le_gender = LabelEncoder()
le_diet = LabelEncoder()
le_country = LabelEncoder()

df_model['gender_enc'] = le_gender.fit_transform(df_model['Gender'])
df_model['diet_enc'] = le_diet.fit_transform(df_model['Diet Type'])
df_model['country_enc'] = le_country.fit_transform(df_model['Country'])

print('New features:')
print(df_model[['sleep_debt', 'screen_overload', 'work_intensity',
               'exercise_enc', 'stress_enc']].head(6))

## 5. Prepare Features & Split

In [ ]:
FEATURES = [
    'Age', 'Sleep Hours', 'Work Hours per Week', 'Screen Time per Day (Hours)',
    'Social Interaction Score', 'Happiness Score',
    'sleep_debt', 'screen_overload', 'work_intensity',
    'exercise_enc', 'stress_enc', 'gender_enc', 'diet_enc', 'country_enc'
]

FEATURE_DISPLAY = [
    'Age', 'Sleep Hours', 'Work Hrs/Week', 'Screen Time/Day',
    'Social Interaction', 'Happiness Score',
    'Sleep Debt', 'Screen Overload', 'Work Intensity',
    'Exercise Level', 'Stress Level', 'Gender', 'Diet', 'Country'
]

le_target = LabelEncoder()
df_model['target'] = le_target.fit_transform(df_model['Mental Health Condition'])

X = df_model[FEATURES]
y = df_model['target']

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'\nTarget classes: {dict(enumerate(le_target.classes_))}')
print(f'\nX: {X.shape}  |  y: {y.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## 6. Baseline: Logistic Regression

Our starting point — a simple linear model. This sets the bar that ensemble models need to beat.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('=== BASELINE: Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'F1 macro : {f1_score(y_test, y_pred_lr, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=le_target.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix — Logistic Regression', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Ensemble Model 1: Random Forest (Bagging)

**How it works:** Trains many decision trees on random subsets of data and features, then takes a majority vote.  
**Why it's better:** Each tree sees different data, so they make different mistakes — averaging them reduces error.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

cv_rf = cross_val_score(rf, X, y, cv=5, scoring='f1_macro', n_jobs=-1)

print('=== ENSEMBLE: Random Forest ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1 macro      : {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'5-Fold CV F1  : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Ensemble Model 2: XGBoost (Boosting)

**How it works:** Builds trees sequentially — each new tree learns from the mistakes of the previous ones.  
**Why it's better:** Focuses effort on hard examples, usually the most accurate ensemble method on tabular data.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

cv_xgb = cross_val_score(xgb_model, X, y, cv=5, scoring='f1_macro', n_jobs=-1)

print('=== ENSEMBLE: XGBoost ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'F1 macro      : {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'5-Fold CV F1  : {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=le_target.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix — XGBoost', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 9. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression (Baseline)', 'Random Forest', 'XGBoost'],
    'Test Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'F1 Macro': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro'),
        f1_score(y_test, y_pred_xgb, average='macro')
    ]
}).round(4)

baseline_f1 = results['F1 Macro'].iloc[0]
results['vs Baseline'] = (results['F1 Macro'] - baseline_f1).round(4)
results['vs Baseline'] = results['vs Baseline'].apply(lambda x: f'+{x:.4f}' if x > 0 else f'{x:.4f}')

print(results.to_string(index=False))

best_idx = results['F1 Macro'].idxmax()
best_name = results.loc[best_idx, 'Model']
print(f'\n>>> Best model: {best_name}')

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(8, 4))
models = results['Model']
f1_scores = results['F1 Macro']
colors = ['#95a5a6', '#2ecc71', '#e67e22']
bars = ax.barh(models, f1_scores, color=colors, edgecolor='white', height=0.5)
ax.set_xlabel('F1 Score (Macro)', fontsize=11)
ax.set_title('Model Comparison — F1 Score', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1)
for bar, val in zip(bars, f1_scores):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 10. Explainability with SHAP

SHAP (SHapley Additive exPlanations) answers: **"Why did the model predict THIS for THIS person?"**  
This is what separates a good ML project from a great one — you don't just predict, you explain.

In [ ]:
# Use best model (XGBoost usually wins on tabular)
best_model = xgb_model

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# shap_values is a list of arrays for multi-class: one per class
print(f'SHAP values type: {type(shap_values)}')
if isinstance(shap_values, list):
    print(f'Number of classes: {len(shap_values)}')
    print(f'Shape per class: {shap_values[0].shape}')
else:
    print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# Global feature importance — which features matter most across ALL classes?
plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURE_DISPLAY,
    class_names=list(le_target.classes_),
    plot_type='bar',
    max_display=14,
    show=True
)

In [ ]:
# Beeswarm plot for top class (e.g. Anxiety, index 0 in sorted classes)
# Shows HOW each feature affects that class prediction
class_idx = list(le_target.classes_).index('Anxiety')
plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values[class_idx],
    X_test,
    feature_names=FEATURE_DISPLAY,
    max_display=14,
    show=True
)

In [ ]:
# Single prediction explanation
sample_idx = 5
sample = X_test.iloc[[sample_idx]]

predicted_class_idx = best_model.predict(sample)[0]
predicted_label = le_target.classes_[predicted_class_idx]
probas = best_model.predict_proba(sample)[0]

print(f'Sample #{sample_idx}')
print(f'Predicted condition: {predicted_label}')
print('Probabilities:')
for cls, prob in sorted(zip(le_target.classes_, probas), key=lambda x: -x[1]):
    bar = '█' * int(prob * 30)
    print(f'  {cls:<12} {bar} {prob:.3f}')

In [ ]:
# Force plot: what pushed this prediction?
shap.force_plot(
    explainer.expected_value[predicted_class_idx],
    shap_values[predicted_class_idx][sample_idx],
    X_test.iloc[sample_idx],
    feature_names=FEATURE_DISPLAY,
    matplotlib=True,
    show=True
)

## 11. Save Best Model

In [ ]:
os.makedirs('../app/model', exist_ok=True)

joblib.dump(best_model,  '../app/model/model.pkl')
joblib.dump(le_target,   '../app/model/label_encoder.pkl')
joblib.dump(FEATURES,    '../app/model/feature_names.pkl')

# Save encoders needed by the Streamlit app
encoders = {
    'le_gender':  le_gender,
    'le_diet':    le_diet,
    'le_country': le_country
}
joblib.dump(encoders, '../app/model/encoders.pkl')

print('Saved:')
for f in ['model.pkl', 'label_encoder.pkl', 'feature_names.pkl', 'encoders.pkl']:
    print(f'  ../app/model/{f}')